# NextBeat — regenerate results.json with the dislike-violation fix
Run all cells top to bottom. Needs no GPU. Takes a while only because of the dataset download (multi-GB), not because of training — this notebook never trains, it only re-evaluates already-trained weights.

**Merge first.** This notebook clones `main` from GitHub. The `run.py evaluate` stage used below (further down) only exists on the `dislike-violation-fix` branch, which is unmerged and will be deleted after merge. Do **not** point the clone at that branch — it is temporary. Instead, merge `dislike-violation-fix` into `main` first, and only run this notebook afterward. Running it before the merge will fail at `run.py evaluate` with `argument stage: invalid choice: 'evaluate'`, **after** the multi-GB dataset download in the download step.

In [ ]:
!git clone https://github.com/falafell99/holberton-project.git
%cd holberton-project
!pip install -q -r requirements.txt

In [ ]:
!python download.py

In [ ]:
import json
manifest = json.load(open('artifacts/manifest.json'))
print('Reproducing seed', manifest['seed'], 'users', manifest['selected_users'], 'catalog', manifest['catalog'])
!python run.py prepare --raw data/raw/multi_event.parquet --out artifacts --users {manifest['selected_users']} --catalog {manifest['catalog']} --seed {manifest['seed']}

In [ ]:
!python run.py evaluate --out artifacts

In [ ]:
for seed_dir in ['experiments/seed_43', 'experiments/seed_44']:
    seed_manifest = json.load(open(f'{seed_dir}/manifest.json'))
    !python run.py prepare --raw data/raw/multi_event.parquet --out {seed_dir} --users {seed_manifest['selected_users']} --catalog {seed_manifest['catalog']} --seed {seed_manifest['seed']}
    !python run.py evaluate --out {seed_dir}

## Download the regenerated results
Download `artifacts/results.json`, `experiments/seed_43/results.json`, and `experiments/seed_44/results.json` from the Colab file browser and replace the copies in your local repo, then commit.

**Scope note for the README.** The NFVR numbers this notebook regenerates are computed against `active_dislikes()`, which scans each user's *entire* history — so NFVR will read `0.000%` here. The live app (`app.py`/`api.py`) only filters against `windowed_active_dislikes()`, scoped to the last 20 events stored in `demo.npz` (that's all the shipped data has). A user who disliked a track more than 20 events ago can still be recommended it live, even though this metric reports 0%. When writing up the README, phrase the NFVR claim as scoped to "within the model's serving/evaluation window," not as an unconditional guarantee, and note explicitly that the live app's filter only looks at the last 20 events.